# Homework 1 — Agentic RAG (LLM Zoomcamp 2026)

Solution du devoir du **Module 1 : Agentic RAG**.

On construit un systeme RAG a partir des **pages de lecons** du cours, puis on le rend *agentique*.

| Question | Sujet |
|----------|-------|
| Q1 | Nombre de pages de lecons |
| Q2 | Indexation + recherche (minsearch) |
| Q3 | RAG + comptage des tokens d'entree |
| Q4 | Chunking |
| Q5 | RAG sur chunks + comparaison des tokens |
| Q6 | Transformation en agent (toyaikit) |

> **Important** : Q3, Q5 et Q6 appellent un LLM. Il faut une cle `OPENAI_API_KEY`
> dans un fichier `.env` (voir cellule Setup). Les questions Q1, Q2 et Q4 ne
> necessitent aucun LLM.

## Setup

Dependances necessaires :

```bash
pip install gitsource minsearch openai toyaikit python-dotenv
```

Pour les questions LLM (Q3, Q5, Q6), creer un fichier `.env` a cote de ce notebook :

```
OPENAI_API_KEY=sk-...
```

In [1]:
from dotenv import load_dotenv, find_dotenv

# usecwd=True : remonte l'arborescence depuis le dossier courant pour trouver le .env
load_dotenv(find_dotenv(usecwd=True))

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

MODEL = "gpt-5.4-mini"

## Preparation — telechargement des pages de lecons

On recupere les pages de lecons directement depuis le depot du cours, en figeant
le commit `8c1834d` pour que tout le monde travaille sur les memes donnees.

Le filtre `/lessons/` garde uniquement les pages de lecons (pas les README, etc.).

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

# Chaque fichier a une methode parse() qui renvoie {filename, content}
documents = [file.parse() for file in files]

print("Champs d'un document :", list(documents[0].keys()))
print("Exemple de filename   :", documents[0]["filename"])

Champs d'un document : ['content', 'filename']
Exemple de filename   : 01-agentic-rag/lessons/01-intro.md


## Q1. Combien de pages de lecons ?

Reponse : **72**

In [3]:
print("Q1 - nombre de pages de lecons :", len(documents))

Q1 - nombre de pages de lecons : 72


## Q2. Indexation et recherche

On indexe avec minsearch : `content` en champ texte, `filename` en champ keyword.

Requete : *How does the agentic loop keep calling the model until it stops?*

Reponse : **`01-agentic-rag/lessons/14-agentic-loop.md`**

In [4]:
index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)

print("Q2 - premier resultat :", results[0]["filename"])

Q2 - premier resultat : 01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG

On construit un assistant RAG sur l'index de Q2. `RAGBase` a ete ecrit pour le
schema FAQ (`section`/`question`/`answer`) ; on l'adapte a notre schema
(`filename`/`content`).

Adaptations apportees :
- `search` utilise notre index ;
- `build_context` concatene `filename` + `content` ;
- `llm` renvoie la **reponse complete** (pas seulement le texte) pour acceder a `usage` ;
- `rag` renvoie un tuple `(answer, usage)`.

On lit ensuite `usage.input_tokens` (tokens d'entree / prompt).

Reponse : **7000** (≈ 7,5k tokens mesures).

> Necessite `OPENAI_API_KEY`.

In [5]:
from openai import OpenAI

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
""".strip()

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()


class RAG:
    """RAG adapte au schema filename/content, qui expose le usage du LLM."""

    def __init__(self, index, llm_client, model=MODEL):
        self.index = index
        self.llm_client = llm_client
        self.model = model

    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(doc["filename"])
            lines.append(doc["content"])
            lines.append("")
        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return PROMPT_TEMPLATE.format(question=query, context=context)

    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": INSTRUCTIONS},
            {"role": "user", "content": prompt},
        ]
        # On renvoie la reponse complete pour pouvoir lire response.usage
        return self.llm_client.responses.create(
            model=self.model,
            input=input_messages,
        )

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response.output_text, response.usage


openai_client = OpenAI()

In [6]:
rag = RAG(index, openai_client)
answer, usage = rag.rag(query)

print("Reponse :", answer[:300], "...\n")
print("Q3 - tokens d'entree (input/prompt) :", usage.input_tokens)

Reponse : It keeps calling the model in a `while True` loop.

Each iteration:
1. Sends the full `messages` history to the model.
2. Checks the response for any `function_call`.
3. Runs the tool and appends the tool result.
4. Sets `has_function_calls = True` if there was a tool call.

Then it stops only when  ...

Q3 - tokens d'entree (input/prompt) : 7110


## Q4. Chunking

On decoupe chaque page en fenetres glissantes de `size=2000` caracteres, avec un
pas de `step=1000` (donc un chevauchement de 1000 caracteres entre chunks).

Reponse : **295**

In [7]:
chunks = chunk_documents(documents, size=2000, step=1000)

print("Q4 - nombre de chunks :", len(chunks))
print("Champs d'un chunk :", list(chunks[0].keys()))

Q4 - nombre de chunks : 295
Champs d'un chunk : ['start', 'content', 'filename']


## Q5. RAG avec chunking

On indexe les chunks (meme schema), on pointe le RAG dessus, et on repose la
meme question. Le contexte envoye est plus petit (chunks de 2000 caracteres au
lieu de pages entieres), donc moins de tokens d'entree.

Reponse : **3× moins** (≈ 7,5k tokens vs ≈ 2,4k tokens).

> Necessite `OPENAI_API_KEY`.

In [8]:
chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

rag_chunks = RAG(chunk_index, openai_client)
answer_c, usage_c = rag_chunks.rag(query)

print("Q3 (pages entieres) input tokens :", usage.input_tokens)
print("Q5 (chunks)         input tokens :", usage_c.input_tokens)
print("Ratio Q3 / Q5 :", round(usage.input_tokens / usage_c.input_tokens, 1), "x moins")

Q3 (pages entieres) input tokens : 7110
Q5 (chunks)         input tokens : 2293
Ratio Q3 / Q5 : 3.1 x moins


## Q6. Transformation en agent

On donne un outil `search` au LLM et on le laisse decider quand (et quoi)
chercher. On utilise `toyaikit`. La fonction `search` a un type hint et une
docstring : toyaikit en deduit automatiquement le schema de l'outil.

On compte combien de fois l'agent appelle `search`.

Reponse : **4** (l'agent decide lui-meme, le nombre varie un peu selon les runs).

> Necessite `OPENAI_API_KEY`.

In [9]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

# Compteur d'appels a l'outil search
search_calls = 0


def search(query: str) -> list[dict]:
    """
    Search the course lessons for chunks matching the given query.
    """
    global search_calls
    search_calls += 1
    return chunk_index.search(query, num_results=5)


agent_tools = Tools()
agent_tools.add_tool(search)  # toyaikit lit le type hint + docstring

agent_instructions = (
    "You're a course teaching assistant. Answer the student's question using "
    "the search tool. Make multiple searches with different keywords before "
    "answering."
)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=agent_instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model=MODEL),
)

In [10]:
search_calls = 0  # remise a zero avant le run

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

print("Q6 - nombre d'appels a search :", search_calls)

-> Response received


-> Response received


Q6 - nombre d'appels a search : 4


## Recapitulatif des reponses

| Question | Reponse |
|----------|---------|
| **Q1** — Nombre de pages | **72** |
| **Q2** — Premier resultat | **`01-agentic-rag/lessons/14-agentic-loop.md`** |
| **Q3** — Tokens d'entree | **7000** |
| **Q4** — Nombre de chunks | **295** |
| **Q5** — Tokens en moins | **3× moins** |
| **Q6** — Appels a search | **4** |

Soumission : https://courses.datatalks.club/llm-zoomcamp-2026/homework/hw1